# multi_strat_046

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (63).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 6 |
| Detected functions | kalman_filter |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
%%black
import pandas as pd

data = pd.read_csv('/content/drive/MyDrive/ipynb/HVStocks/merged_file.csv')
data

In [ ]:
import pandas as pd

# Remove rows where the 'ticker' column contains 'ULTRACEMCO' or 'BARTIRTEL'
#data = data[data['Ticker'].isin(['ADANIENT', 'BAJFINANCE', 'SBIN', 'SBILIFE', 'MARUTI', 'TATAMOTORS'])]
data = data[data['Ticker'].isin(['RELIANCE'])]

# Remove rows where any column contains 0 or NaN
data = data[(data != 0).all(axis=1)].dropna()

data


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Example Kalman Filter function (Assumed to be pre-defined)
def kalman_filter(prices):
    n = len(prices)
    predicted_prices = np.zeros(n)
    process_variance = 1e-7
    estimated_measurement_variance = 3
    posteri_estimate = prices[0]
    posteri_error_estimate = 1.0

    for t in range(n):
        prior_estimate = posteri_estimate
        prior_error_estimate = posteri_error_estimate + process_variance
        blending_factor = prior_error_estimate / (prior_error_estimate + estimated_measurement_variance)
        posteri_estimate = prior_estimate + blending_factor * (prices[t] - prior_estimate)
        posteri_error_estimate = (1 - blending_factor) * prior_error_estimate
        predicted_prices[t] = posteri_estimate

    return predicted_prices

# Example Dataset
data['OHLC4'] = data['close']  # Simplified for example
#data.set_index('datetime', inplace=True)

# Apply Kalman Filter
data['Kalman_Price'] = kalman_filter(data['close'])

# Feature Engineering
data['kalman_residual'] = data['close'] - data['Kalman_Price']
data['ma_20'] = data['close'].rolling(21).mean()
data['ma_50'] = data['close'].rolling(60).mean()
data['rsi'] = 100 - (100 / (1 + data['close'].pct_change().rolling(14).mean()))

# Define Target (Buy: 1, Sell: -1, Hold: 0)
data['target'] = np.where(data['close'].shift(-1) > data['close'], 1, -1)

# Prepare Dataset for ML Model
features = ['OHLC4', 'kalman_residual', 'ma_20', 'ma_50', 'rsi']
data.dropna(inplace=True)  # Drop rows with NaN values from rolling calculations
X = data[features]
y = data['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

# Use Model to Predict Signals
data['predicted_signal'] = model.predict(X[features])

# Backtest Logic with Brokerage Fee
initial_capital = 100000
capital = initial_capital  # Use capital directly instead of portfolio
position = 0  # Current position (1 for long, -1 for short, 0 for none)
brokerage_fee = 0.001  # 0.1% brokerage fee
tradebook = []
cumulative_pnl = 0  # Initialize cumulative PNL

for i in range(len(data)):
    current_price = data['close'].iloc[i]
    signal = data['predicted_signal'].iloc[i]
    current_time = data.index[i]

    # Enter Long Position (Buy)
    if signal == 1 and position == 0:
        position = capital / current_price  # Buy with full capital
        entry_price = current_price
        entry_time = current_time
        tradebook.append({
            'Entry Price': entry_price, 'Exit Price': None, 'Entry Time': entry_time, 'Exit Time': None,
            'PNL': 0, 'Total PNL': capital - initial_capital, 'Cumulative PNL': cumulative_pnl
        })

    # Enter Short Position (Sell)
    elif signal == -1 and position == 0:
        position = -capital / current_price  # Short with full capital
        entry_price = current_price
        entry_time = current_time
        tradebook.append({
            'Entry Price': entry_price, 'Exit Price': None, 'Entry Time': entry_time, 'Exit Time': None,
            'PNL': 0, 'Total PNL': capital - initial_capital, 'Cumulative PNL': cumulative_pnl
        })

    # Exit Long Position
    elif signal == -1 and position > 0:
        pnl = (current_price - entry_price) * position
        capital += pnl
        capital -= brokerage_fee * position * current_price  # Deduct brokerage fee
        cumulative_pnl += pnl
        position = 0
        exit_price = current_price
        exit_time = current_time
        tradebook.append({
            'Entry Price': entry_price, 'Exit Price': exit_price, 'Entry Time': entry_time, 'Exit Time': exit_time,
            'PNL': pnl - brokerage_fee * position * current_price, 'Total PNL': capital - initial_capital, 'Cumulative PNL': cumulative_pnl
        })

    # Exit Short Position
    elif signal == 1 and position < 0:
        pnl = (entry_price - current_price) * abs(position)
        capital += pnl
        capital -= brokerage_fee * abs(position) * current_price  # Deduct brokerage fee
        cumulative_pnl += pnl
        position = 0
        exit_price = current_price
        exit_time = current_time
        tradebook.append({
            'Entry Price': entry_price, 'Exit Price': exit_price, 'Entry Time': entry_time, 'Exit Time': exit_time,
            'PNL': pnl - brokerage_fee * abs(position) * current_price, 'Total PNL': capital - initial_capital, 'Cumulative PNL': cumulative_pnl
        })

# Evaluate Results
final_capital_value = capital + (position * current_price if position != 0 else 0)
print(f"Final Capital Value: {final_capital_value:.2f}")

# Create Tradebook DataFrame and Print
tradebook_df = pd.DataFrame(tradebook)
print(tradebook_df)

# Print Final Cumulative PnL
print(f"Final Cumulative PnL: {cumulative_pnl:.2f}")
